# 11. Создание цикла обучения

Видео: [11. Создание цикла обучения](https://www.youtube.com/watch?v=QZ-vKDAaThY&list=PLBP4Q3FNSLK2rtGPBsK-aMAetYj-8yg_1&index=12)

## О чём урок

Данные, модель, loss и Adam уже есть. Здесь — **петля**, которая их связывает:

```text
эпоха:
  train:  model.train() → батчи → forward → loss → zero_grad → backward → step
  val:    model.eval() + torch.no_grad() → батчи → forward → loss / метрика  (без step)
```

> **Конспект:** петля и backprop — [`d.4.1`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.1 Полносвязные нейронные сети/d.4.1 Полносвязные нейронные сети.md) **§11**; loss/метрика — урок `08/`; модель — `09/`–`10/`. Карта — [`0_как_учить_pytorch.md`](../0_как_учить_pytorch.md).

**Отличие от автора.** В видео метки иногда делают one-hot. С `nn.CrossEntropyLoss` нужны **индексы классов** `(B,)`, как в уроке 8 — так и делаем.  
У нас в модели уже есть `Flatten` → `reshape` батча в `(B, 784)` не нужен.

**Как смотреть.** ~20 мин — **кусками**:

| блок | о чём | где roughly |
|:---|:---|:---|
| 1 | один шаг: zero_grad → backward → step | начало цикла |
| 2 | accuracy через `argmax` | метрика |
| 3 | одна train-эпоха по DataLoader | train-цикл |
| 4 | val-эпоха: `eval` + `no_grad` | val-цикл |
| 5 | несколько эпох + списки истории | конец + графики у автора |

**Как решать.** Замени `...` → проверка. Решения — `11_цикл_обучения_решения.ipynb` после попытки.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(0)
device = torch.device("cpu")  # GPU позже; логика цикла та же


def need(cond, msg):
    assert cond, msg


def need_tensor(x, name):
    assert isinstance(x, torch.Tensor), f"{name} — пока не тензор (замени ... на код)"


print("torch", torch.__version__, "| device:", device)

## 1. Один шаг обучения

> **Конспект:** [`d.4.1`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.1 Полносвязные нейронные сети/d.4.1 Полносвязные нейронные сети.md) **§11.1** (петля) и **§11.2** (backprop vs оптимизатор).
>
> Там словами то же, что здесь в трёх строках кода:
> 1. батч → forward → лосс;
> 2. **backward** считает градиенты `∂L/∂w` (правило цепочки / autograd) — веса ещё **не** меняет;
> 3. **оптимизатор** (Adam) делает `w ← …` по этим градиентам.
>
> В таблице §11.2 прямо: `loss.backward()` ≠ `optimizer.step()`. Зачем градиент и цепочка — [`d.0`](../../../все мои конспекты в ynote/d.Машинное обучение/d.0 Градиент и правило цепочки.md). Игрушечный шаг Adam — урок `08/` задание 5.1.

Порядок **всегда** такой:

```python
opt.zero_grad()   # 1. стереть старые градиенты
loss.backward()   # 2. посчитать новые .grad
opt.step()        # 3. сдвинуть веса
```

Без `zero_grad` градиенты **накапливаются** между шагами.

In [ ]:
# 1.1 Один шаг Adam на крошечной сети
torch.manual_seed(0)
model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 3)).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
criterion = nn.CrossEntropyLoss()

xb = torch.randn(16, 4, device=device)
yb = torch.randint(0, 3, (16,), device=device)

model.train()
logits = model(xb)
loss_before = criterion(logits, yb)

opt.zero_grad()
loss_before.backward()
opt.step()

logits2 = model(xb)
loss_after = criterion(logits2, yb)

print("loss:", round(loss_before.item(), 4), "→", round(loss_after.item(), 4))

In [ ]:
need(loss_after.item() < loss_before.item(), "после step loss на том же батче должен упасть")
print("1.1 — верно")

## 2. Accuracy на батче

Логиты `(B, C)` → класс = `argmax(dim=1)` → сравнить с `y` → доля верных.

```python
pred = logits.argmax(dim=1)
acc = (pred == y).float().mean()
```

Softmax для argmax **не нужен** (урок 8).

In [ ]:
# 2.1 accuracy через argmax
logits = torch.tensor([[2.0, 0.1, -1.0], [0.0, 3.0, 1.0], [1.0, 1.5, 0.2]])
y = torch.tensor([0, 1, 0])

pred = logits.argmax(dim=1)
correct = (pred == y).sum().item()
acc = (pred == y).float().mean()

print("pred:", pred.tolist(), "| acc:", float(acc))

In [ ]:
need_tensor(pred, "pred")
need(pred.tolist() == [0, 1, 1], f"pred={pred.tolist()}")
need(correct == 2, f"correct={correct}")
need(abs(float(acc) - 2 / 3) < 1e-5, f"acc={acc}")
print("2.1 — верно")

## 3. Одна train-эпоха

Эпоха = один проход по **всем** батчам `train_loader`.

Внутри: `model.train()`, для каждого батча — forward → loss → zero_grad → backward → step.  
Средний loss эпохи: сумма `loss.item()` / число батчей (или взвешенно по размеру батча — у автора часто простой mean по батчам).

In [ ]:
# 3.1 Одна эпоха по DataLoader
torch.manual_seed(0)
X = torch.randn(64, 4)
y = torch.randint(0, 3, (64,))
loader = DataLoader(TensorDataset(X, y), batch_size=16, shuffle=True)

model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 3))
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
criterion = nn.CrossEntropyLoss()

model.train()
batch_losses = []
for xb, yb in loader:
    logits = model(xb)
    loss = criterion(logits, yb)
    opt.zero_grad()
    loss.backward()
    opt.step()
    batch_losses.append(loss.item())

train_loss = sum(batch_losses) / len(batch_losses)
print("train_loss:", round(train_loss, 4), "| batches:", len(batch_losses))

In [ ]:
need(len(batch_losses) == 4, f"батчей должно быть 64/16=4, а не {len(batch_losses)}")
need(isinstance(train_loss, float) and train_loss > 0, "train_loss — среднее loss.item()")
print("3.1 — верно")

## 4. Val-эпоха: `eval` + `no_grad`

На валидации веса **не** трогаем:

```python
model.eval()
with torch.no_grad():
    for xb, yb in val_loader:
        ...
```

- `eval()` — режим Dropout/BN (привычка);
- `no_grad()` — не строить граф → быстрее, меньше памяти;
- **нет** `zero_grad` / `backward` / `step`.

In [ ]:
# 4.1 Val: loss + accuracy без обновления весов
torch.manual_seed(0)
model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 3))
criterion = nn.CrossEntropyLoss()
w_before = model[0].weight.detach().clone()

X = torch.randn(32, 4)
y = torch.randint(0, 3, (32,))
val_loader = DataLoader(TensorDataset(X, y), batch_size=16)

model.eval()
total_loss = 0.0
total_correct = 0
n = 0
with torch.no_grad():
    for xb, yb in val_loader:
        logits = model(xb)
        loss = criterion(logits, yb)
        total_loss += loss.item()
        pred = logits.argmax(dim=1)
        total_correct += (pred == yb).sum().item()
        n += yb.size(0)

val_loss = total_loss / len(val_loader)
val_acc = total_correct / n
w_after = model[0].weight.detach().clone()

print("val_loss:", round(val_loss, 4), "| val_acc:", round(val_acc, 4))
print("weights unchanged:", torch.equal(w_before, w_after))

In [ ]:
need(torch.equal(w_before, w_after), "на val веса не должны меняться (нет step)")
need(n == 32, f"n={n}")
need(0.0 <= val_acc <= 1.0, f"val_acc={val_acc}")
print("4.1 — верно")

## 5. Несколько эпох + история

Снаружи — цикл по эпохам. Внутри каждой: train-проход, потом val-проход.  
В списки кладём средние loss/acc — потом графики (как у автора) и диагностика переобучения (train↓ val↑).

Ожидание на игрушечных данных: train_loss к концу меньше, чем в начале.

In [ ]:
# 5.1 5 эпох: train → val, сохранить историю
torch.manual_seed(0)
Xtr = torch.randn(80, 4)
ytr = torch.randint(0, 3, (80,))
Xva = torch.randn(40, 4)
yva = torch.randint(0, 3, (40,))
train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=16, shuffle=True)
val_loader = DataLoader(TensorDataset(Xva, yva), batch_size=16)

model = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 3))
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
criterion = nn.CrossEntropyLoss()

history = {"train_loss": [], "val_loss": [], "val_acc": []}
EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    losses = []
    for xb, yb in train_loader:
        logits = model(xb)
        loss = criterion(logits, yb)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
    history["train_loss"].append(sum(losses) / len(losses))

    model.eval()
    v_losses = []
    correct = 0
    n = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            logits = model(xb)
            loss = criterion(logits, yb)
            v_losses.append(loss.item())
            pred = logits.argmax(dim=1)
            correct += (pred == yb).sum().item()
            n += yb.size(0)
    history["val_loss"].append(sum(v_losses) / len(v_losses))
    history["val_acc"].append(correct / n)

print("train_loss:", [round(x, 3) for x in history["train_loss"]])
print("val_acc:", [round(x, 3) for x in history["val_acc"]])

In [ ]:
need(len(history["train_loss"]) == 5, "5 эпох → 5 записей")
need(history["train_loss"][-1] < history["train_loss"][0], "train_loss к концу должен упасть")
need(all(0.0 <= a <= 1.0 for a in history["val_acc"]), "val_acc в [0, 1]")
print("5.1 — верно")

## 6. Что мы сделали

| кусок | смысл |
|:---|:---|
| zero_grad → backward → step | один шаг обучения |
| argmax → accuracy | метрика классификации |
| цикл по loader | эпоха train |
| eval + no_grad | эпоха val без обновления весов |
| for epoch | история для графиков / переобучения |

Дальше: раздел **«Видео 11»** в `project/fashion_mnist.ipynb` — тот же цикл на Fashion и квадратах.  
Scheduler — видео 12; сохранение весов — 13; EarlyStopping — 14.